# 03. Model Evaluation & SHAP Interpretability Analysis
### Track: Data Science (Peng et al., 2023 Section 4.6)

**Referensi Ilmiah:**
> **Peng, K., Peng, Y., & Li, W. (2023).** *Research on customer churn prediction and model interpretability analysis.* **PLOS ONE**, 18(12), e0289724.
> File PDF referensi: `notebooks/02_prediction/referensi_churn_xgboost.pdf`.

**Tujuan Notebook:**
1. Menguji model produksi `xgb_churn_model.joblib` yang telah dilatih dengan **SMOTEENN + GA-XGBoost**.
2. Menerapkan interpretasi model **Global SHAP** (Summary Plot, Feature Importance Bar Plot, Dependence Plot).
3. Menerapkan interpretasi model **Lokal SHAP** (Waterfall Plot untuk pelanggan loyal vs pelanggan yang churn).
4. Menjalankan *Prescriptive Loop*: Mencetak skor risiko retensi bagi seluruh pelanggan aktif untuk tim CRM.


In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score, confusion_matrix
import shap

PROJECT_ROOT = Path.cwd().resolve().parents[1] if Path.cwd().name == "02_prediction" else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.telco_churn.core.loader import load_processed_data, MODELS_DIR
from src.telco_churn.prediction.inference import get_predictor, score_customers

sns.set_theme(style="whitegrid", font="sans-serif")
plt.rcParams["figure.dpi"] = 120
print("Setup & SHAP imports completed.")


## 1. Memuat Model Produksi & Data Uji
Memuat pipeline lengkap yang tersimpan di `deliverables/prediction/models/xgb_churn_model.joblib`.


In [ ]:
predictor = get_predictor()
pipeline = predictor.pipeline
print("Model Pipeline Loaded from:", predictor.model_path)

df = load_processed_data()
y = (df["churn"] == "Yes").astype(int)
X = df.drop(columns=["churn", "customer_id"], errors="ignore")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

y_proba = pipeline.predict_proba(X_test)[:, 1]
y_pred = pipeline.predict(X_test)

print(f"ROC-AUC Test: {roc_auc_score(y_test, y_proba):.4f}")
print(f"PR-AUC Test:  {average_precision_score(y_test, y_proba):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Retained (No)', 'Churn (Yes)']))


## 2. SHAP Global Feature Impact (Paper Fig 11 & 12)
Sesuai dengan Bagian 4.6.1 dari paper *Peng et al. (2023)*:
- **SHAP Summary Plot (Beeswarm):** Menampilkan distribusi nilai fitur dan dampaknya terhadap probabilitas churn.
- Titik merah = nilai fitur tinggi, titik biru = nilai fitur rendah.
- Titik di sebelah kanan garis tengah (SHAP > 0) meningkatkan probabilitas churn.


In [ ]:
# Ekstraksi model XGBoost dan preprocessor dari pipeline
xgb_model = pipeline.named_steps["classifier"]
feature_eng = pipeline.named_steps["domain_features"]
preprocessor = pipeline.named_steps["preprocessor"]

X_test_engineered = feature_eng.transform(X_test)
X_test_transformed = preprocessor.transform(X_test_engineered)

# Mendapatkan nama-nama fitur hasil ColumnTransformer
num_cols = preprocessor.named_transformers_["num"].feature_names_in_.tolist()
cat_cols = preprocessor.named_transformers_["cat"].named_steps["onehot"].get_feature_names_out().tolist()
all_feature_names = num_cols + cat_cols

# Menghitung SHAP values menggunakan TreeExplainer
explainer = shap.TreeExplainer(xgb_model)
sample_size = 300
shap_values = explainer.shap_values(X_test_transformed[:sample_size])

plt.figure(figsize=(10, 6.5))
shap.summary_plot(shap_values, X_test_transformed[:sample_size], feature_names=all_feature_names, show=False)
plt.title("SHAP Global Feature Impact on Churn (Paper Fig 11 Methodology)", fontsize=13, fontweight="bold", pad=12)
plt.tight_layout()
plt.show()


## 3. SHAP Feature Importance Ranking (Paper Fig 12 & 13)
Memplot rata-rata nilai absolut SHAP: $\text{mean}(|\text{SHAP value}|)$.


In [ ]:
plt.figure(figsize=(9, 5))
shap.summary_plot(shap_values, X_test_transformed[:sample_size], feature_names=all_feature_names, plot_type="bar", show=False)
plt.title("SHAP Feature Importance Ranking (Paper Fig 12 Methodology)", fontsize=13, fontweight="bold", pad=12)
plt.tight_layout()
plt.show()


> **Interpretasi Global SHAP:**
> 1. `tenure`: Nilai tenure rendah (titik biru) memiliki SHAP value positif besar (mendorong keputusan churn). Pelanggan baru adalah penyumbang churn utama.
> 2. `contract_Month-to-month`: Sangat kuat mendorong nilai prediksi churn ke arah positif.
> 3. `monthly_charges` & `total_charges`: Tagihan bulanan yang tinggi memperbesar tendensi pelanggan untuk menghentikan langganan.
> 4. `fiber_no_tech_support` & `payment_method_Electronic check`: Fitur domain yang secara konsisten memiliki bobot positif pada risiko churn.


## 4. SHAP Feature Dependence Plot (Paper Fig 16)
Menganalisis efek interaksi non-linear antara masa berlangganan (`tenure`) dan tagihan bulanan (`monthly_charges`).


In [ ]:
# Menemukan indeks kolom tenure dan monthly_charges
tenure_idx = all_feature_names.index("tenure")
monthly_idx = all_feature_names.index("monthly_charges")

plt.figure(figsize=(8.5, 5))
shap.dependence_plot(
    tenure_idx,
    shap_values,
    X_test_transformed[:sample_size],
    feature_names=all_feature_names,
    interaction_index=monthly_idx,
    show=False
)
plt.title("SHAP Dependence Plot: Tenure vs Monthly Charges Interaction", fontsize=12, fontweight="bold", pad=12)
plt.tight_layout()
plt.show()


## 5. SHAP Local Interpretation: Waterfall Plots (Paper Fig 14, 15, 17)
Sesuai Bagian 4.6.2 paper *Peng et al. (2023)*, kita menganalisis alasan model memprediksi pelanggan individu:
- Kasus A: Pelanggan Bertahan (*Non-Churn Customer*)
- Kasus B: Pelanggan Berisiko Churn Tinggi (*High-Risk Churn Customer*)


In [ ]:
# Membuat Explanation object untuk visualisasi waterfall
exp = shap.Explanation(
    values=shap_values,
    base_values=explainer.expected_value,
    data=X_test_transformed[:sample_size],
    feature_names=all_feature_names
)

# 1. Contoh Pelanggan Non-Churn (Kasus A)
retained_idx = np.where(y_pred[:sample_size] == 0)[0][0]
plt.figure(figsize=(9, 5))
shap.plots.waterfall(exp[retained_idx], show=False)
plt.title(f"SHAP Waterfall Plot: Non-Churn Customer (Sample #{retained_idx})", fontsize=12, fontweight="bold", pad=12)
plt.tight_layout()
plt.show()


In [ ]:
# 2. Contoh Pelanggan Churn (Kasus B)
churn_idx = np.where(y_pred[:sample_size] == 1)[0][0]
plt.figure(figsize=(9, 5))
shap.plots.waterfall(exp[churn_idx], show=False)
plt.title(f"SHAP Waterfall Plot: High-Risk Churn Customer (Sample #{churn_idx})", fontsize=12, fontweight="bold", pad=12)
plt.tight_layout()
plt.show()


## 6. Prescriptive Retention Strategy (Umpan Balik ke Tim Bisnis)
Memanfaatkan model prediktif untuk menilai seluruh basis pelanggan aktif saat ini dan memberikan rekomendasi tindakan spesifik berdasarkan faktor pendorong SHAP.


In [ ]:
scored_customers = score_customers(df)
top_risk = scored_customers[scored_customers["churn"] == "No"].sort_values(
    by="churn_probability", ascending=False
)[["customer_id", "tenure", "contract", "payment_method", "monthly_charges", "churn_probability", "risk_level", "recommendation"]].head(10)

print(f"Total Pelanggan Aktif Berskor: {len(scored_customers):,} pelanggan")
top_risk


## Ringkasan Akhir
- **Metodologi Paper Terpenuhi:** Mengimplementasikan alur prapemrosesan, penyeimbangan data dengan SMOTEENN, benchmarking model, tuning XGBoost, dan interpretasi SHAP global serta lokal.
- **Transparansi Model:** Model black-box XGBoost berhasil dibuka dan dijelaskan secara intuitif untuk kebutuhan pemangku kepentingan bisnis dan tim retensi lapangan.
